# Dataset 1 – Hand-Written Digits Recognition
**Implementation:** Vishal Hanuman  
**Algorithms:** k-NN (HW1) · Random Forest (HW3) · Neural Network (HW4)  
**Evaluation:** Stratified 10-fold CV · Accuracy · Macro F1

In [ ]:
import sys, os, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['git', 'clone', 'https://github.com/vishalhanuman14/classical-ml-benchmark.git'], check=True)
    os.chdir('classical-ml-benchmark')

if '.' not in sys.path:
    sys.path.insert(0, '.')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn import datasets

from algorithms.vishal import knn, random_forest, neural_network, utils

np.random.seed(42)
os.makedirs('results', exist_ok=True)

# ── Load & normalize ──────────────────────────────────────────────────────────
digits      = datasets.load_digits(return_X_y=True)
X_raw, y_int = digits[0], digits[1]
y           = y_int.astype(str)          # string labels for kNN / RF
X_norm, _   = utils.normalize(X_raw, X_raw)

print(f"Instances: {len(y)}  Features: {X_raw.shape[1]}  Classes: {np.unique(y)}")


## k-NN – Hyperparameter Sweep (k)

In [ ]:
K_VALUES = [1, 3, 5, 7, 11, 15, 21, 31]
knn_results = {}

for k in K_VALUES:
    def model_fn(Xtr, ytr, Xte, k=k):
        Xtr_n, Xte_n = utils.normalize(Xtr, Xte)
        return knn.predict(Xtr_n, ytr, Xte_n, k)

    acc, f1 = utils.cross_validate(model_fn, X_raw, y, pos_label=y[0])
    knn_results[k] = (acc, f1)
    print(f"k={k:>3}  acc={acc:.4f}  f1={f1:.4f}")


In [ ]:
best_k = max(knn_results, key=lambda k: knn_results[k][1])
print(f"Best k={best_k}  acc={knn_results[best_k][0]:.4f}  f1={knn_results[best_k][1]:.4f}")

ks = list(knn_results.keys())
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ks, [knn_results[k][0] for k in ks], '-o', label='Accuracy')
ax.plot(ks, [knn_results[k][1] for k in ks], '-s', label='F1')
ax.set_xlabel('k'); ax.set_ylabel('Score')
ax.set_title('k-NN: Accuracy & F1 vs k  (Digits, 10-fold CV)')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig('results/digits_knn_k_sweep.png', dpi=150); plt.show()


## Random Forest – Hyperparameter Sweep (ntree)

In [ ]:
feature_types = ['num'] * X_norm.shape[1]
NTREE_VALUES  = [5, 10, 20, 30, 50, 75, 100, 150]
rf_results    = {}

for nt in NTREE_VALUES:
    def model_fn(Xtr, ytr, Xte, nt=nt):
        trees = random_forest.train(Xtr, ytr, feature_types, nt)
        return random_forest.predict(trees, Xte)

    acc, f1 = utils.cross_validate(model_fn, X_norm, y, pos_label=y[0])
    rf_results[nt] = (acc, f1)
    print(f"ntree={nt:>4}  acc={acc:.4f}  f1={f1:.4f}")


In [ ]:
best_nt = max(rf_results, key=lambda n: rf_results[n][1])
print(f"Best ntree={best_nt}  acc={rf_results[best_nt][0]:.4f}  f1={rf_results[best_nt][1]:.4f}")

nts = list(rf_results.keys())
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(nts, [rf_results[n][0] for n in nts], '-o', label='Accuracy')
ax.plot(nts, [rf_results[n][1] for n in nts], '-s', label='F1')
ax.set_xlabel('ntree'); ax.set_ylabel('Score')
ax.set_title('Random Forest: Accuracy & F1 vs ntree  (Digits, 10-fold CV)')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig('results/digits_rf_ntree_sweep.png', dpi=150); plt.show()


## Neural Network – Hyperparameter Sweep (architecture × lambda)

Input: 64 features → hidden layers → 10 output neurons (one-hot).  
6 architecture + regularisation combinations evaluated with 10-fold CV.

In [ ]:
# One-hot Y for NN training (10 classes)
N_CLASSES = 10
Y_oh = neural_network.one_hot(y_int, N_CLASSES)  # shape (1797, 10)

# (hidden_layers_tuple, lambda)
NN_CONFIGS = [
    ((32,),      0.0),
    ((64,),      0.0),
    ((32, 16),   0.0),
    ((64, 32),   0.0),
    ((32,),      0.1),
    ((64, 32),   0.1),
]

nn_results = {}
folds = utils.stratified_k_fold(y, k=10)

for arch, lam in NN_CONFIGS:
    accs, f1s = [], []
    for i in range(10):
        test_idx  = np.array(folds[i])
        train_idx = np.array([idx for j in range(10) if j != i for idx in folds[j]])

        Xtr, Xte   = X_norm[train_idx], X_norm[test_idx]
        Ytr        = Y_oh[train_idx]
        y_test_int = y_int[test_idx]

        layer_sizes = [64] + list(arch) + [N_CLASSES]
        theta, _    = neural_network.train(Xtr, Ytr, layer_sizes,
                                           lam=lam, alpha=0.3,
                                           epochs=300, batch_size=32)
        y_pred = neural_network.predict(theta, Xte)
        acc, f1 = utils.compute_metrics(y_test_int, y_pred, pos_label=0)
        accs.append(acc); f1s.append(f1)

    nn_results[(arch, lam)] = (float(np.mean(accs)), float(np.mean(f1s)))
    print(f"arch={str(arch):<12} lam={lam}  acc={np.mean(accs):.4f}  f1={np.mean(f1s):.4f}")


In [ ]:
best_nn_key = max(nn_results, key=lambda k: nn_results[k][1])
best_arch, best_lam = best_nn_key
print(f"Best config: arch={best_arch}  lam={best_lam}")
print(f"  acc={nn_results[best_nn_key][0]:.4f}  f1={nn_results[best_nn_key][1]:.4f}")

labels = [f"{str(a)}\nlam={l}" for a, l in nn_results]
accs   = [nn_results[k][0] for k in nn_results]
f1s    = [nn_results[k][1] for k in nn_results]
x      = np.arange(len(labels))

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - 0.2, accs, 0.4, label='Accuracy')
ax.bar(x + 0.2, f1s,  0.4, label='F1')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Score')
ax.set_title('Neural Network: Accuracy & F1 per config  (Digits, 10-fold CV)')
ax.legend(); ax.grid(alpha=0.3, axis='y'); plt.tight_layout()
plt.savefig('results/digits_nn_config_sweep.png', dpi=150); plt.show()


### Neural Network – Learning Curve (J vs training instances)

Train the best architecture on increasing subsets of data; measure cost J on a fixed held-out test set.

In [ ]:
# 80/20 split for the learning curve
split      = int(0.8 * len(y_int))
order      = np.random.permutation(len(y_int))
tr_idx, te_idx = order[:split], order[split:]

Xtr_lc, Xte_lc = X_norm[tr_idx], X_norm[te_idx]
Ytr_lc = Y_oh[tr_idx]
Yte_lc = Y_oh[te_idx]

step    = max(50, split // 14)
sizes   = list(range(50, split + 1, step))
if sizes[-1] != split:
    sizes.append(split)

lc_costs = []
layer_sizes_best = [64] + list(best_arch) + [N_CLASSES]

for n in sizes:
    theta_lc, _ = neural_network.train(Xtr_lc[:n], Ytr_lc[:n], layer_sizes_best,
                                        lam=best_lam, alpha=0.3,
                                        epochs=300, batch_size=32)
    J, _, _ = neural_network.cost(Xte_lc, Yte_lc, theta_lc, best_lam)
    lc_costs.append(J)
    print(f"n={n:>5}  J={J:.4f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sizes, lc_costs, '-o')
ax.set_xlabel('Number of training instances')
ax.set_ylabel('Cost J (test set)')
ax.set_title(f'NN Learning Curve  arch={best_arch}  lam={best_lam}  (Digits)')
ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig('results/digits_nn_learning_curve.png', dpi=150); plt.show()


## Summary – Dataset 1 (Digits)

In [ ]:
print(f"{'Algorithm':<25} {'Best Hyperparameter':<22} {'Accuracy':>10} {'F1 Score':>10}")
print('-' * 72)
print(f"{'k-NN':<25} {'k='+str(best_k):<22} {knn_results[best_k][0]:>10.4f} {knn_results[best_k][1]:>10.4f}")
print(f"{'Random Forest':<25} {'ntree='+str(best_nt):<22} {rf_results[best_nt][0]:>10.4f} {rf_results[best_nt][1]:>10.4f}")
print(f"{'Neural Network':<25} {str(best_arch)+' lam='+str(best_lam):<22} {nn_results[best_nn_key][0]:>10.4f} {nn_results[best_nn_key][1]:>10.4f}")
